In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0001.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0002.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0003.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0004.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0005.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0006.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0007.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0008.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0009.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0010.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0011.png
/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR/0

In [2]:
!git clone https://github.com/Algolzw/daclip-uir.git /kaggle/working/daclip-uir
%cd /kaggle/working/daclip-uir
!ls

Cloning into '/kaggle/working/daclip-uir'...
remote: Enumerating objects: 532, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 532 (delta 93), reused 76 (delta 76), pack-reused 399 (from 1)
Receiving objects: 100% (532/532), 17.41 MiB | 21.80 MiB/s, done.
Resolving deltas: 100% (184/184), done.
/kaggle/working/daclip-uir
cog.yaml  figs	  LICENSE     README.md		scripts
da-clip   images  predict.py  requirements.txt	universal-image-restoration


In [3]:
!pip install -q open_clip_torch ftfy regex lmdb pyyaml tqdm opencv-python scikit-image lpips huggingface_hub
!cat requirements.txt   # check for anything missing, but skip torch/torchvision pins

absl-py==2.0.0
cachetools==5.3.1
certifi==2023.7.22
charset-normalizer==3.3.0
cmake==3.27.6
einops==0.7.0
ema-pytorch==0.2.3
filelock==3.12.4
ftfy==6.1.1
google-auth==2.23.2
google-auth-oauthlib==1.0.0
grpcio==1.59.0
idna==3.4
importlib-metadata==6.8.0
Jinja2==3.1.2
joblib==1.3.2
lit==17.0.1
lmdb==1.4.1
Markdown==3.4.4
MarkupSafe==2.1.3
mpmath==1.3.0
networkx==3.1
numpy==1.24.4
nvidia-cublas-cu11==11.10.3.66
nvidia-cuda-cupti-cu11==11.7.101
nvidia-cuda-nvrtc-cu11==11.7.99
nvidia-cuda-runtime-cu11==11.7.99
nvidia-cudnn-cu11==8.5.0.96
nvidia-cufft-cu11==10.9.0.58
nvidia-curand-cu11==10.2.10.91
nvidia-cusolver-cu11==11.4.0.1
nvidia-cusparse-cu11==11.7.4.91
nvidia-nccl-cu11==2.14.3
nvidia-nvtx-cu11==11.7.91
oauthlib==3.2.2
opencv-python==4.8.1.78
Pillow==10.0.1
protobuf==4.24.3
pyasn1==0.5.0
pyasn1-modules==0.3.0
PyYAML==6.0.1
regex==2023.8.8
requests==2.31.0
requests-oauthlib==1.3.1
rsa==4.9
scikit-learn==1.3.1
scipy==1.10.1
sympy==1.12
tensorboard==2.14.0
tensorboard-data-server==0.7.1
t

In [4]:
import os
from huggingface_hub import hf_hub_download

os.makedirs("universal-image-restoration/pretrained", exist_ok=True)
for f in ["daclip_ViT-B-32.pt", "universal-ir.pth"]:
    hf_hub_download("weblzw/daclip-uir-ViT-B-32-irsde", f,
                    local_dir="universal-image-restoration/pretrained")
!ls universal-image-restoration/pretrained

daclip_ViT-B-32.pt: reconstructing file:   0%|          |  0.00B / 1.74GB            

daclip_ViT-B-32.pt: downloading bytes:           |  0.00B            

universal-ir.pth: reconstructing file:   0%|          |  0.00B /  196MB            

universal-ir.pth: downloading bytes:           |  0.00B            

daclip_ViT-B-32.pt  universal-ir.pth


In [12]:
!ls /kaggle/input/datasets/

joe1995


In [14]:
!ls /kaggle/input
!ls /kaggle/input/datasets/joe1995/div2k-dataset/

datasets
div2k_train_hr	DIV2K_train_HR	div2k_valid_hr	DIV2K_valid_HR


In [16]:
import os

base_path = '/kaggle/input/datasets/'
print("Searching for image folders...")

for root, dirs, files in os.walk(base_path):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count > 0:
        print(f"Found {png_count} images in: {root}")

Searching for image folders...
Found 800 images in: /kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR
Found 100 images in: /kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_valid_HR/DIV2K_valid_HR
Found 800 images in: /kaggle/input/datasets/joe1995/div2k-dataset/div2k_train_hr/DIV2K_train_HR
Found 100 images in: /kaggle/input/datasets/joe1995/div2k-dataset/div2k_valid_hr/DIV2K_valid_HR


In [17]:
import os
import cv2
import glob

# Using the exact nested path for the training set
input_dir = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'
output_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
scale_factor = 4  # Change to 2 or 3 if needed

os.makedirs(output_dir, exist_ok=True)

# Fetch all HR image paths
hr_images = glob.glob(os.path.join(input_dir, '*.png'))
print(f"Found {len(hr_images)} images. Starting compression...")

for img_path in hr_images:
    filename = os.path.basename(img_path)
    img = cv2.imread(img_path)
    
    if img is None:
        continue
        
    # Calculate new dimensions
    height, width = img.shape[:2]
    new_width = width // scale_factor
    new_height = height // scale_factor
    
    # Downsample using bicubic interpolation
    lr_img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)
    
    # Save the compressed image
    output_path = os.path.join(output_dir, filename)
    cv2.imwrite(output_path, lr_img)

print(f"Compression complete. LR images saved to {output_dir}")

Found 800 images. Starting compression...
Compression complete. LR images saved to /kaggle/working/DIV2K_train_LR_bicubic/X4


In [18]:
import yaml
import os

config_path = "universal-image-restoration/config/daclip-sde/options/test.yml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Point the dataloader to your compressed DIV2K dataset
config['datasets']['test']['dataroot_lq'] = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
# If you want to calculate metrics (PSNR/SSIM), point to the original HR Kaggle input path
config['datasets']['test']['dataroot_gt'] = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'

# Ensure it points to your downloaded pretrained weights
config['path']['pretrain_model_G'] = 'universal-image-restoration/pretrained/universal-ir.pth'
config['path']['daclip'] = 'universal-image-restoration/pretrained/daclip_ViT-B-32.pt'

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print("Updated test.yml with Kaggle paths!")

KeyError: 'test'

In [19]:
import yaml
import os

config_path = "universal-image-restoration/config/daclip-sde/options/test.yml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Get the actual test dataset key (usually 'test_1' or 'val')
test_keys = list(config['datasets'].keys())

for key in test_keys:
    # Update all test dataset paths found in the config
    config['datasets'][key]['dataroot_lq'] = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
    config['datasets'][key]['dataroot_gt'] = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'

# Ensure it points to your downloaded pretrained weights
config['path']['pretrain_model_G'] = 'universal-image-restoration/pretrained/universal-ir.pth'
config['path']['daclip'] = 'universal-image-restoration/pretrained/daclip_ViT-B-32.pt'

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print(f"Updated test.yml successfully! Replaced paths for: {test_keys}")

Updated test.yml successfully! Replaced paths for: ['test1']


In [20]:
%cd universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
export CUDA_VISIBLE_DEVICES=0
OrderedDict({'dataroot_GT': 'datasets/universal/deg_type/GT', 'dataroot_LQ': 'datasets/universal/deg_type/LQ', 'dataroot_gt': '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR', 'dataroot_lq': '/kaggle/working/DIV2K_train_LR_bicubic/X4', 'mode': 'LQGT', 'name': 'Test'})
rm: cannot remove './result': No such file or directory
26-10-30 17:56:06.956 - INFO:   datasets:[
    test1:[
      dataroot_GT: datasets/universal/deg_type/GT
      dataroot_LQ: datasets/universal/deg_type/LQ
      dataroot_gt: /kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR
      dataroot_lq: /

In [21]:
import yaml

config_path = "/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/options/test.yml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

test_keys = list(config['datasets'].keys())

for key in test_keys:
    # Update the correct uppercase keys
    config['datasets'][key]['dataroot_LQ'] = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
    config['datasets'][key]['dataroot_GT'] = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'
    
    # Remove the incorrect lowercase keys
    config['datasets'][key].pop('dataroot_lq', None)
    config['datasets'][key].pop('dataroot_gt', None)

# Set absolute paths for pretrained models
config['path']['pretrain_model_G'] = '/kaggle/working/daclip-uir/universal-image-restoration/pretrained/universal-ir.pth'
config['path']['daclip'] = '/kaggle/working/daclip-uir/universal-image-restoration/pretrained/daclip_ViT-B-32.pt'

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print("Fixed YAML config with uppercase keys!")

Fixed YAML config with uppercase keys!


In [23]:
!pip install ema_pytorch

In [29]:
!sed -i 's/checkpoint = torch.load(checkpoint_path, map_location=map_location)/checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)/g' /kaggle/working/daclip-uir/universal-image-restoration/open_clip/factory.py

In [30]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
export CUDA_VISIBLE_DEVICES=0
OrderedDict({'dataroot_GT': '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR', 'dataroot_LQ': '/kaggle/working/DIV2K_train_LR_bicubic/X4', 'mode': 'LQGT', 'name': 'Test'})
26-10-30 18:02:53.775 - INFO:   datasets:[
    test1:[
      dataroot_GT: /kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR
      dataroot_LQ: /kaggle/working/DIV2K_train_LR_bicubic/X4
      mode: LQGT
      name: Test
      phase: test1
      scale: 1
      distortion: ['motion-blurry', 'hazy', 'jpeg-compressed', 'low-light', 'noisy', 'raindrop', 'rainy', 'shadowed', 'snowy', 'uncompleted']
   

In [31]:
import os
import random
import glob

lr_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
all_images = sorted(glob.glob(os.path.join(lr_dir, '*.png')))

# Seed for reproducibility so you get the same 300 images if you rerun this
random.seed(42)

if len(all_images) > 300:
    # Randomly select 300 images to keep
    images_to_keep = set(random.sample(all_images, 300))
    
    deleted_count = 0
    for img_path in all_images:
        if img_path not in images_to_keep:
            os.remove(img_path)
            deleted_count += 1
            
    print(f"Deleted {deleted_count} images. Exactly 300 diverse images remain in the folder.")
else:
    print(f"The folder currently has {len(all_images)} images. No deletion needed.")

Deleted 500 images. Exactly 300 diverse images remain in the folder.


In [32]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
export CUDA_VISIBLE_DEVICES=0
OrderedDict({'dataroot_GT': '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR', 'dataroot_LQ': '/kaggle/working/DIV2K_train_LR_bicubic/X4', 'mode': 'LQGT', 'name': 'Test'})
26-10-30 18:09:06.494 - INFO:   datasets:[
    test1:[
      dataroot_GT: /kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR
      dataroot_LQ: /kaggle/working/DIV2K_train_LR_bicubic/X4
      mode: LQGT
      name: Test
      phase: test1
      scale: 1
      distortion: ['motion-blurry', 'hazy', 'jpeg-compressed', 'low-light', 'noisy', 'raindrop', 'rainy', 'shadowed', 'snowy', 'uncompleted']
   

In [33]:
import os
import shutil
import glob
import yaml

lr_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
gt_original_dir = '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR'
gt_subset_dir = '/kaggle/working/DIV2K_train_HR_subset'

os.makedirs(gt_subset_dir, exist_ok=True)

# Fetch the 300 remaining LR image filenames
lr_images = glob.glob(os.path.join(lr_dir, '*.png'))

# Copy the corresponding HR images to a new working folder
for lr_path in lr_images:
    filename = os.path.basename(lr_path)
    src_gt = os.path.join(gt_original_dir, filename)
    dst_gt = os.path.join(gt_subset_dir, filename)
    if os.path.exists(src_gt):
        shutil.copy(src_gt, dst_gt)

print(f"Copied {len(os.listdir(gt_subset_dir))} matching GT images to {gt_subset_dir}")

# Update test.yml to point to the new 300-image GT folder
config_path = "/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/options/test.yml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

for key in list(config['datasets'].keys()):
    config['datasets'][key]['dataroot_GT'] = gt_subset_dir

with open(config_path, 'w') as f:
    yaml.dump(config, f)
    
print("Updated config to use the new matching subset!")

Copied 300 matching GT images to /kaggle/working/DIV2K_train_HR_subset
Updated config to use the new matching subset!


In [34]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
export CUDA_VISIBLE_DEVICES=0
OrderedDict({'dataroot_GT': '/kaggle/working/DIV2K_train_HR_subset', 'dataroot_LQ': '/kaggle/working/DIV2K_train_LR_bicubic/X4', 'mode': 'LQGT', 'name': 'Test'})
26-10-30 18:10:47.985 - INFO:   datasets:[
    test1:[
      dataroot_GT: /kaggle/working/DIV2K_train_HR_subset
      dataroot_LQ: /kaggle/working/DIV2K_train_LR_bicubic/X4
      mode: LQGT
      name: Test
      phase: test1
      scale: 1
      distortion: ['motion-blurry', 'hazy', 'jpeg-compressed', 'low-light', 'noisy', 'raindrop', 'rainy', 'shadowed', 'snowy', 'uncompleted']
      data_type: img
    ]
  ]
  degradation:[
    noise_type: G
    scale: 4

In [35]:
import os

test_py_path = '/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/test.py'

with open(test_py_path, 'r') as f:
    code = f.read()

# The exact replacement code that forces resizing before scoring
patch_code = """
            if cropped_sr_img.shape != cropped_gt_img.shape:
                import cv2
                cropped_gt_img = cv2.resize(cropped_gt_img, (cropped_sr_img.shape[1], cropped_sr_img.shape[0]))
            psnr = util.calculate_psnr(cropped_sr_img * 255, cropped_gt_img * 255)
"""

# Replace the specific PSNR calculation line
code = code.replace(
    "psnr = util.calculate_psnr(cropped_sr_img * 255, cropped_gt_img * 255)", 
    patch_code.strip()
)

with open(test_py_path, 'w') as f:
    f.write(code)

print("test.py has been successfully patched to resize ground truth images!")

test.py has been successfully patched to resize ground truth images!


In [36]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
export CUDA_VISIBLE_DEVICES=0
OrderedDict({'dataroot_GT': '/kaggle/working/DIV2K_train_HR_subset', 'dataroot_LQ': '/kaggle/working/DIV2K_train_LR_bicubic/X4', 'mode': 'LQGT', 'name': 'Test'})
26-10-30 18:12:40.316 - INFO:   datasets:[
    test1:[
      dataroot_GT: /kaggle/working/DIV2K_train_HR_subset
      dataroot_LQ: /kaggle/working/DIV2K_train_LR_bicubic/X4
      mode: LQGT
      name: Test
      phase: test1
      scale: 1
      distortion: ['motion-blurry', 'hazy', 'jpeg-compressed', 'low-light', 'noisy', 'raindrop', 'rainy', 'shadowed', 'snowy', 'uncompleted']
      data_type: img
    ]
  ]
  degradation:[
    noise_type: G
    scale: 4

In [37]:
test_py_path = '/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde/test.py'

with open(test_py_path, 'r') as f:
    code = f.read()

# Resize the PyTorch GT tensor to match SR_img's spatial dimensions (H, W) before LPIPS
old_lpips = "lp_score = lpips_fn("
new_lpips = """if GT.shape[-2:] != SR_img.shape[-2:]:
                GT = torch.nn.functional.interpolate(GT, size=SR_img.shape[-2:], mode='bilinear', align_corners=False)
            lp_score = lpips_fn("""

if "torch.nn.functional.interpolate(GT" not in code:
    code = code.replace(old_lpips, new_lpips)
    with open(test_py_path, 'w') as f:
        f.write(code)
    print("Patched LPIPS tensor shape mismatch in test.py!")
else:
    print("Patch already applied.")

Patched LPIPS tensor shape mismatch in test.py!


In [38]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
export CUDA_VISIBLE_DEVICES=0
OrderedDict({'dataroot_GT': '/kaggle/working/DIV2K_train_HR_subset', 'dataroot_LQ': '/kaggle/working/DIV2K_train_LR_bicubic/X4', 'mode': 'LQGT', 'name': 'Test'})
26-10-30 18:14:35.558 - INFO:   datasets:[
    test1:[
      dataroot_GT: /kaggle/working/DIV2K_train_HR_subset
      dataroot_LQ: /kaggle/working/DIV2K_train_LR_bicubic/X4
      mode: LQGT
      name: Test
      phase: test1
      scale: 1
      distortion: ['motion-blurry', 'hazy', 'jpeg-compressed', 'low-light', 'noisy', 'raindrop', 'rainy', 'shadowed', 'snowy', 'uncompleted']
      data_type: img
    ]
  ]
  degradation:[
    noise_type: G
    scale: 4

In [39]:
import os
import cv2
import glob

lq_dir = '/kaggle/working/DIV2K_train_LR_bicubic/X4'
gt_dir = '/kaggle/working/DIV2K_train_HR_subset'

lq_files = sorted(glob.glob(os.path.join(lq_dir, '*.png')))

for lq_path in lq_files:
    fname = os.path.basename(lq_path)
    gt_path = os.path.join(gt_dir, fname)
    
    if os.path.exists(gt_path):
        lq_img = cv2.imread(lq_path)
        gt_img = cv2.imread(gt_path)
        
        # Step 1 (Option A): Resize GT to match LQ's exact 1:1 dimensions
        h, w = lq_img.shape[:2]
        gt_resized = cv2.resize(gt_img, (w, h), interpolation=cv2.INTER_AREA)
        cv2.imwrite(gt_path, gt_resized)
        
        # Step 2 (Option B): Apply Paper's JPEG Compression (Quality=10) to create true LQ
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 10]
        _, encimg = cv2.imencode('.jpg', gt_resized, encode_param)
        jpeg_degraded_img = cv2.imdecode(encimg, 1)
        cv2.imwrite(lq_path, jpeg_degraded_img)

print(f"Prepared {len(lq_files)} aligned 1:1 GT and JPEG-degraded (QF=10) LQ pairs!")

Prepared 300 aligned 1:1 GT and JPEG-degraded (QF=10) LQ pairs!


In [40]:
%cd /kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
!python test.py -opt=options/test.yml

/kaggle/working/daclip-uir/universal-image-restoration/config/daclip-sde
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
export CUDA_VISIBLE_DEVICES=0
OrderedDict({'dataroot_GT': '/kaggle/working/DIV2K_train_HR_subset', 'dataroot_LQ': '/kaggle/working/DIV2K_train_LR_bicubic/X4', 'mode': 'LQGT', 'name': 'Test'})
26-10-30 18:20:54.634 - INFO:   datasets:[
    test1:[
      dataroot_GT: /kaggle/working/DIV2K_train_HR_subset
      dataroot_LQ: /kaggle/working/DIV2K_train_LR_bicubic/X4
      mode: LQGT
      name: Test
      phase: test1
      scale: 1
      distortion: ['motion-blurry', 'hazy', 'jpeg-compressed', 'low-light', 'noisy', 'raindrop', 'rainy', 'shadowed', 'snowy', 'uncompleted']
      data_type: img
    ]
  ]
  degradation:[
    noise_type: G
    scale: 4